In [38]:
import math
from pyspark.sql import SparkSession
spark = SparkSession.builder.appName('NaiveBayes').getOrCreate()

In [39]:
train_data = spark.read.csv('train_data.csv', header=True, inferSchema=True)
train_data_rdd = train_data.rdd
num_features = len(train_data_rdd.first()) - 1

In [40]:
# Map features and labels
train_data_mapped = train_data_rdd.map(lambda x: (x[:-1], int(x[-1])))

In [41]:
# Calculate class priors
class_counts = train_data_mapped.map(lambda x: (x[1], 1)).reduceByKey(lambda x, y: x + y)
total_count = train_data_rdd.count()
class_priors = class_counts.map(lambda x: (x[0], x[1] / float(total_count)))

In [42]:
# Calculate feature counts by class
feature_counts_by_class = train_data_mapped.flatMap(lambda x: [(i, x[0][i], x[1]) for i in range(len(x[0]))])
feature_counts_by_class = feature_counts_by_class.map(lambda x: ((x[0], x[1], x[2]), 1)).reduceByKey(lambda x, y: x + y)

In [43]:
# Sort feature counts
sorted_features = feature_counts_by_class.sortBy(lambda x: (x[0][0], x[0][2]))

In [44]:
# Convert class counts to dictionary
class_counts_list = class_counts.collect()
class_counts_dict = dict(class_counts_list)

In [45]:
# Calculate conditional probabilities
conditional_probs = feature_counts_by_class.map(lambda x: ((x[0][0], x[0][1], x[0][2]), (x[1], class_counts_dict[x[0][2]])))
conditional_probs = conditional_probs.sortBy(lambda x: (x[0][0], x[0][2]))
conditional_probs = conditional_probs.mapValues(lambda x: (x[0] + 1) / (x[1] + num_features))

In [46]:
# Sort conditional probabilities
sorted_probs = conditional_probs.sortBy(lambda x: (x[0][0], x[0][2]))

In [47]:
# Convert probabilities to dictionary
prob_dict = sorted_probs.collectAsMap()

In [48]:
# Rearrange probabilities for faster lookup
prob_dict = sorted_probs.map(lambda x: (x[0][2], ((x[0][0], x[0][1]), x[1]))).groupByKey().mapValues(list).collectAsMap()

In [49]:
# Convert probabilities to have feature as key in inner dictionary
prob_dict = sorted_probs.map(lambda x: (x[0][2], ((x[0][0], x[0][1]), x[1]))).groupByKey().mapValues(list).collectAsMap()

In [50]:
# Convert class priors to dictionary
class_priors = class_priors.collectAsMap()

In [51]:
# Rearrange probability dictionary
new_dict = {}

for key, value in prob_dict.items():
    new_dict[key] = {}
    for sub_value in value:
        inner_key = sub_value[0][0]
        inner_value = (sub_value[0][1], sub_value[1])
        if inner_key in new_dict[key]:
            new_dict[key][inner_key].append(inner_value)
        else:
            new_dict[key][inner_key] = [inner_value]

In [52]:
# Classify testing data
def classify(data):
    features = data[0]
    actual_label = data[1]
    predicted_label = None
    max_posterior = float('-inf')

    for class_label in class_priors.keys():
        posterior = math.log(class_priors[class_label])
        for feature, feature_value in enumerate(features):
            for value, prob in new_dict[class_label][feature]:
                if value == feature_value:
                    posterior += math.log(prob)
                    break
        if posterior > max_posterior:
            max_posterior = posterior
            predicted_label = class_label

    return (actual_label, predicted_label)

In [53]:
# Read testing data
test_data = spark.read.csv('test_data.csv', header=True, inferSchema=True).rdd
test_data_mapped = test_data.map(lambda x: (x[:-1], int(x[-1])))
predictions = test_data_mapped.map(classify)

In [54]:
# Calculate accuracy
correct_predictions = predictions.filter(lambda x: x[0] == x[1]).count()
accuracy = correct_predictions / float(test_data.count())
print("Accuracy:", accuracy)

# Stop Spark session
spark.stop()

Accuracy: 0.07977448281985261
